# K-Means Clustering

K-Means is an unsupervised algorithm that partitions data into **K clusters** by iteratively assigning points to the nearest centroid and updating centroids to the cluster mean.

**Dataset:** Mall Customers (simulated) — Annual Income vs Spending Score.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, '.')
from kmeans import kmeans
np.random.seed(42)
print("Imports complete")

## Load & Explore the Data

In [ ]:
# Simulate mall customer data: Annual Income vs Spending Score
np.random.seed(42)
groups = [
    (20, 5, 20, 7, 40),   # low income, low spend
    (55, 8, 50, 8, 50),   # mid income, mid spend
    (85, 6, 80, 6, 40),   # high income, high spend
    (30, 6, 80, 8, 35),   # low income, high spend
    (75, 7, 15, 7, 35),   # high income, low spend
]
income = np.concatenate([np.random.normal(mu_i, s_i, n) for mu_i,s_i,_,_,n in groups])
spend  = np.concatenate([np.random.normal(mu_s, s_s, n) for _,_,mu_s,s_s,n in groups])
X = np.clip(np.column_stack([income, spend]), 1, 100)

print(f"Dataset shape: {X.shape}")
print(f"Income  — mean: {X[:,0].mean():.1f}, std: {X[:,0].std():.1f}")
print(f"Spending— mean: {X[:,1].mean():.1f}, std: {X[:,1].std():.1f}")

plt.figure(figsize=(7,5))
plt.scatter(X[:,0], X[:,1], alpha=0.6, color='steelblue', edgecolors='white', linewidths=0.3, s=40)
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Mall Customers — Raw Data")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Elbow Method — Choosing K

We compute inertia for K=1..10 and look for the "elbow" where adding more clusters yields diminishing returns.

In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    m = kmeans(n_clusters=k, max_iter=300, random_state=42)
    m.fit(X)
    inertias.append(m.inertia_)

plt.figure(figsize=(8,4))
plt.plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2, markersize=7)
plt.axvline(5, color='tomato', linestyle='--', alpha=0.8, label='Elbow: K=5')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
print("Inertias:", [round(v,1) for v in inertias])

## Fit K-Means (K=5)

In [ ]:
K = 5
model = kmeans(n_clusters=K, max_iter=300, random_state=42)
labels = model.fit_predict(X)

print(f"Converged in {model.n_iter_} iterations")
print(f"Final inertia: {model.inertia_:.2f}")
print(f"Cluster sizes: {[(labels==k).sum() for k in range(K)]}")

## Visualise Clusters

In [ ]:
COLORS = ['#e63946','#457b9d','#2a9d8f','#e9c46a','#a8dadc']

plt.figure(figsize=(9,6))
for k in range(K):
    mask = labels == k
    plt.scatter(X[mask,0], X[mask,1], color=COLORS[k],
                label=f'Cluster {k}', alpha=0.75, edgecolors='white', linewidths=0.3, s=50)
plt.scatter(model.centroids[:,0], model.centroids[:,1],
            color='black', marker='X', s=180, zorder=5, label='Centroids')
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("K-Means Clustering — K=5")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cluster Profiles

In [ ]:
print(f"{'Cluster':>9} | {'N':>5} | {'Avg Income':>12} | {'Avg Spending':>13}")
print("-" * 48)
for k in range(K):
    mask = labels == k
    print(f"  Cluster {k} | {mask.sum():>5} | {X[mask,0].mean():>11.1f}  | {X[mask,1].mean():>12.1f}")
print(f"\nModel score (neg. inertia): {model.score(X):.2f}")

## Key Takeaways

- K-Means partitions customers into 5 actionable spending segments.
- The **elbow method** guides K selection without ground-truth labels.
- Centroids represent the average member of each cluster.
- **Limitation:** K-Means assumes roughly spherical clusters and is sensitive to initialisation.
